# 2.3 Download GICS Classifications

This notebook downloads GICS classifications for the stock universe and saves them separately in `data/gics`.

GICS has four levels: a 2-digit sector, 4-digit industry group, 6-digit industry, and 8-digit sub-industry. The public constituent source contains current S&P 500 ticker-to-sub-industry assignments. Former constituents may therefore remain unmatched; they are saved in a separate review file instead of being assigned an unreliable current classification.

## Important limitation: current GICS is used as a historical approximation

A company’s GICS classification can change when its main business changes, after a merger or spin-off, or when the GICS system itself is revised. The free source used here provides the **current** classification, not the classification that was known on each historical date.

For now, this project makes the simplifying assumption that each company’s current GICS classification also applies to its past observations. This keeps the workflow straightforward, but it can create look-ahead bias: a backtest may group two stocks together using information that was assigned only later. The approximation is most limited for companies that changed their business substantially or were reclassified during the study period. Therefore, results that depend on industry matching should be treated as preliminary.

A future version should use point-in-time GICS history. That dataset would contain a ticker or permanent company identifier, an effective start date, an effective end date, and the GICS code for each period. The backtest could then select the classification that was valid on each date. Sources such as WRDS/Compustat HGIC or FactSet can provide this history, but they require licensed access. No historical classifications are invented in this notebook; until such a source is available, the current-code approximation is kept and documented here.

## 1. Setup

Each companies's sub industry tag is downloaded through the Github dataset: https://github.com/datasets/s-and-p-500-companies
The GICS mapping for each sub industry is downloaded through the Github dataset: https://gist.github.com/uknj/c9bcf66ab379a35fcc8758f9a6c86ceb

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == 'notebooks':
    PROJECT_FOLDER = PROJECT_FOLDER.parent

STOCK_FOLDER = PROJECT_FOLDER / 'data' / 'Filter_historical'
GICS_FOLDER = PROJECT_FOLDER / 'data' / 'gics'
GICS_FOLDER.mkdir(parents=True, exist_ok=True)

CONSTITUENTS_URL = (
    'https://raw.githubusercontent.com/datasets/'
    's-and-p-500-companies/main/data/constituents.csv'
)
GICS_STRUCTURE_URL = (
    'https://gist.githubusercontent.com/uknj/'
    'c9bcf66ab379a35fcc8758f9a6c86ceb/raw/'
    'GICS%20Mappings%20-%20March%202023%20Update.csv'
)

## 2. Load the project ticker universe

Each CSV filename in `Filter_historical` represents one ticker used by the project.

In [2]:
tickers = sorted(path.stem.upper() for path in STOCK_FOLDER.glob('*.csv'))
ticker_df = pd.DataFrame({'Ticker': tickers})

print(f'Project tickers: {len(ticker_df)}')
ticker_df.head()

Project tickers: 616


,Ticker
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV


## 3. Download current company classifications

The current S&P 500 table provides each company’s ticker, name, GICS sector, and GICS sub-industry. It gives us the sub-industry, which we later used to match the 8-digit GICS code.

In [3]:
constituents = pd.read_csv(CONSTITUENTS_URL)
constituents = constituents.rename(columns={
    'Symbol': 'Ticker',
    'Security': 'Company',
})
constituents['Ticker'] = (
    constituents['Ticker'].str.replace('.', '-', regex=False).str.upper()
)
constituents = constituents[[
    'Ticker', 'Company', 'GICS Sector', 'GICS Sub-Industry'
]]

print(f'Current constituent rows: {len(constituents)}')
constituents.head()

Current constituent rows: 503


,Ticker,Company,GICS Sector,GICS Sub-Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


## 4. Download the GICS code hierarchy

This table connects each sub-industry name to its 2-, 4-, 6-, and 8-digit GICS codes and descriptions.

In [4]:
gics_structure = pd.read_csv(
    GICS_STRUCTURE_URL,
    dtype={
        'Sector Code': 'string',
        'Industry Group Code': 'string',
        'Industry Code': 'string',
        'Sub-Industry Code': 'string',
    },
)

GICS_COLUMNS = [
    'Sector Code', 'Sector',
    'Industry Group Code', 'Industry Group',
    'Industry Code', 'Industry',
    'Sub-Industry Code', 'Sub-Industry',
]
gics_structure = gics_structure[GICS_COLUMNS].drop_duplicates()
gics_structure.to_csv(GICS_FOLDER / 'gics_structure.csv', index=False)

print(f'GICS sub-industries: {len(gics_structure)}')
gics_structure.head()

GICS sub-industries: 163


,Sector Code,Sector,Industry Group Code,Industry Group,Industry Code,Industry,Sub-Industry Code,Sub-Industry
0,10,Energy,1010,Energy,101010,Energy Equipment & Services,10101010,Oil & Gas Drilling
1,10,Energy,1010,Energy,101010,Energy Equipment & Services,10101020,Oil & Gas Equipment & Services
2,10,Energy,1010,Energy,101020,"Oil, Gas & Consumable Fuels",10102010,Integrated Oil & Gas
3,10,Energy,1010,Energy,101020,"Oil, Gas & Consumable Fuels",10102020,Oil & Gas Exploration & Production
4,10,Energy,1010,Energy,101020,"Oil, Gas & Consumable Fuels",10102030,Oil & Gas Refining & Marketing


## 5. Match tickers to all four GICS levels

Small punctuation differences are removed from sub-industry names before matching. The saved codes remain strings so leading zeroes can never be lost.
Merge constitutents and gics_structure according to sub-industry.
Tickers in the database that matches stored in matched_cigs
Tickers in the database that does not match is stored in unmatched_cigs.

In [5]:
def clean_name(values):
    return (
        values.str.lower()
        .str.replace('&', 'and', regex=False)
        .str.replace(r'[^a-z0-9]+', '', regex=True)
    )

constituents['match_name'] = clean_name(constituents['GICS Sub-Industry'])
gics_structure['match_name'] = clean_name(gics_structure['Sub-Industry'])

current_gics = constituents.merge(
    gics_structure,
    on='match_name',
    how='left',
    validate='many_to_one',
)
ticker_gics = ticker_df.merge(current_gics, on='Ticker', how='left')

OUTPUT_COLUMNS = [
    'Ticker', 'Company',
    'Sector Code', 'Sector',
    'Industry Group Code', 'Industry Group',
    'Industry Code', 'Industry',
    'Sub-Industry Code', 'Sub-Industry',
]
ticker_gics = ticker_gics[OUTPUT_COLUMNS]

matched_gics = ticker_gics.dropna(subset=['Sub-Industry Code']).copy()
unmatched_tickers = ticker_gics[ticker_gics['Sub-Industry Code'].isna()][
    ['Ticker']
].copy()

## 6. Save and review

Existing dataset rows are preserved, including researched sector/industry-group assignments and intentionally blank finer levels. Downloads add only previously unseen tickers. `unmatched_tickers.csv` lists universe tickers missing sector or industry group. Research sources are recorded separately in `data/gics/gics2_gics4_sources.csv`.


In [ ]:
# Preserve every saved row, including intentionally blank industry/sub-industry fields.
gics_path = GICS_FOLDER / 'ticker_gics.csv'
if gics_path.exists():
    saved_gics = pd.read_csv(gics_path, dtype='string', keep_default_na=False)
    if saved_gics['Ticker'].duplicated().any():
        raise ValueError('Saved GICS mapping contains duplicate tickers')
    new_gics = matched_gics.loc[~matched_gics['Ticker'].isin(saved_gics['Ticker'])]
    matched_gics = pd.concat([saved_gics[OUTPUT_COLUMNS], new_gics], ignore_index=True)

has_sector_and_group = (
    matched_gics[['Sector Code', 'Industry Group Code']]
    .fillna('').astype(str).apply(lambda column: column.str.strip().ne('')).all(axis=1)
)
classified_tickers = matched_gics.loc[has_sector_and_group, 'Ticker']
unmatched_tickers = ticker_df.loc[
    ~ticker_df['Ticker'].isin(classified_tickers), ['Ticker']
].copy()

matched_gics.to_csv(gics_path, index=False)
unmatched_tickers.to_csv(GICS_FOLDER / 'unmatched_tickers.csv', index=False)

print(f'Saved ticker rows: {len(matched_gics)}')
print(f'Unmatched sector/industry-group tickers: {len(unmatched_tickers)}')
print(f'Saved to: {GICS_FOLDER}')
unmatched_tickers
